# Graph Visualization with topologic_fast

This notebook demonstrates graph visualization techniques using topologic_fast.

We will:
1. Create a CellComplex (a 3D topological structure)
2. Derive its dual graph (connecting cells through shared faces)
3. Visualize the graph using Plotly
4. Analyze graph properties like centrality

**Note:** This is adapted from topologicpy's GraphViz tutorial. The original uses GraphViz library for export/visualization. Here we use Plotly for interactive visualization.

In [ ]:
# Import topologic_fast and visualization libraries
import topologic_fast as tf
import plotly.graph_objects as go
import numpy as np

## Create a Sample CellComplex

We'll create a prism-like CellComplex with multiple cells arranged in a grid pattern. This will give us an interesting graph structure to visualize.

In [ ]:
# Create a grid of cells (3x3x2 building)
cells = []
cell_names = []
cell_colors = []

color_palette = [
    '#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', 
    '#FFEAA7', '#DDA0DD', '#98D8C8', '#F7DC6F',
    '#BB8FCE', '#85C1E9', '#F8B500', '#00CED1',
    '#FF69B4', '#32CD32', '#FFD700', '#9370DB',
    '#20B2AA', '#FF7F50'
]

cell_idx = 0
for k in range(2):  # 2 floors
    for j in range(3):  # 3 rows
        for i in range(3):  # 3 columns
            # Create a box cell
            cell = tf.Cell.Box(i * 2, j * 2, k * 3, 2, 2, 3)
            cells.append(cell)
            cell_names.append(f"Cell {cell_idx + 1}")
            cell_colors.append(color_palette[cell_idx % len(color_palette)])
            cell_idx += 1

# Create the CellComplex
cc = tf.CellComplex.ByCells(cells)

print(f"CellComplex created with {cc.NumCells()} cells")
print(f"Total volume: {cc.Volume():.1f} cubic units")

## Derive the Dual Graph

The dual graph represents:
- **Vertices**: Centroids of each cell
- **Edges**: Connections between cells that share a face

In [ ]:
# Create dual graph from the CellComplex
graph = tf.Graph.ByTopology(cc)

print(f"Graph Statistics:")
print(f"  Vertices (cells): {graph.Order()}")
print(f"  Edges (shared faces): {graph.Size()}")
print(f"  Density: {graph.Density():.3f}")
print(f"  Diameter: {graph.Diameter()} steps")
print(f"  Is Connected: {graph.IsConnected()}")

## Visualize the Graph with Plotly (2D Projection)

We'll create a 2D network-style visualization of the graph.

In [ ]:
def visualize_graph_2d(graph, node_labels=None, node_colors=None, title="Graph Visualization"):
    """
    Create a 2D visualization of a graph using Plotly.
    Uses spring layout positioning for nodes.
    """
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    # Get vertex coordinates (using x, y from 3D coords)
    coords = [v.Coordinates() for v in vertices]
    
    fig = go.Figure()
    
    # Draw edges
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                mode='lines',
                line=dict(color='rgba(150,150,150,0.6)', width=2),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    
    # Calculate node degrees for sizing
    degrees = [graph.VertexDegree(v) for v in vertices]
    sizes = [15 + d * 3 for d in degrees]
    
    colors = node_colors if node_colors else ['#1f77b4'] * len(vertices)
    labels = node_labels if node_labels else [f"Node {i+1}" for i in range(len(vertices))]
    
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='markers+text',
        marker=dict(
            size=sizes,
            color=colors,
            line=dict(color='black', width=1)
        ),
        text=[str(i+1) for i in range(len(vertices))],
        textposition='middle center',
        textfont=dict(size=10, color='white'),
        hovertext=[f"{labels[i]}<br>Degree: {degrees[i]}" for i in range(len(vertices))],
        hoverinfo='text',
        showlegend=False
    ))
    
    fig.update_layout(
        title=title,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor='x'),
        width=800,
        height=600,
        plot_bgcolor='white'
    )
    
    return fig

# Visualize
fig = visualize_graph_2d(graph, cell_names, cell_colors, "CellComplex Dual Graph (Top View)")
fig.show()

## 3D Graph Visualization with CellComplex

Now let's visualize both the CellComplex geometry and its dual graph together in 3D.

In [ ]:
def visualize_cellcomplex_with_graph(cellcomplex, graph, cell_colors, cell_names):
    """
    Create a 3D visualization showing both the CellComplex geometry
    and its dual graph overlaid.
    """
    fig = go.Figure()
    
    cells = cellcomplex.Cells()
    
    # Draw cells
    for i, cell in enumerate(cells):
        faces = cell.Faces()
        color = cell_colors[i] if i < len(cell_colors) else '#808080'
        
        for j, face in enumerate(faces):
            vertices = face.Vertices()
            coords = [v.Coordinates() for v in vertices]
            
            if len(coords) >= 3:
                x = [c[0] for c in coords]
                y = [c[1] for c in coords]
                z = [c[2] for c in coords]
                
                fig.add_trace(go.Mesh3d(
                    x=x, y=y, z=z,
                    color=color,
                    opacity=0.3,
                    alphahull=0,
                    name=cell_names[i] if i < len(cell_names) else f"Cell {i+1}",
                    showlegend=(j == 0)
                ))
                
                # Add wireframe edges
                for k in range(len(coords)):
                    p1 = coords[k]
                    p2 = coords[(k + 1) % len(coords)]
                    fig.add_trace(go.Scatter3d(
                        x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
                        mode='lines',
                        line=dict(color='black', width=1),
                        showlegend=False,
                        hoverinfo='skip'
                    ))
    
    # Draw graph edges (red lines connecting cell centroids)
    graph_edges = graph.Edges()
    for edge in graph_edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='red', width=5),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw graph vertices (red spheres at cell centroids)
    graph_vertices = graph.Vertices()
    vertex_coords = [v.Coordinates() for v in graph_vertices]
    x = [c[0] for c in vertex_coords]
    y = [c[1] for c in vertex_coords]
    z = [c[2] for c in vertex_coords]
    
    degrees = [graph.VertexDegree(v) for v in graph_vertices]
    
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(
            size=[5 + d for d in degrees],
            color='red',
            line=dict(color='darkred', width=1)
        ),
        name='Graph Vertices',
        hovertext=[f"{cell_names[i]}<br>Degree: {degrees[i]}" for i in range(len(graph_vertices))],
        hoverinfo='text'
    ))
    
    fig.update_layout(
        title='CellComplex with Dual Graph Overlay',
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=900,
        height=700,
        legend=dict(x=1.02, y=1)
    )
    
    return fig

fig_3d = visualize_cellcomplex_with_graph(cc, graph, cell_colors, cell_names)
fig_3d.show()

## Analyze Graph Properties

Let's analyze various graph metrics including vertex degrees and connectivity patterns.

In [ ]:
# Analyze vertex degrees
graph_vertices = graph.Vertices()
degrees = [graph.VertexDegree(v) for v in graph_vertices]

print("Vertex Degree Analysis:")
print("=" * 50)
for i, (name, degree) in enumerate(zip(cell_names, degrees)):
    print(f"  {name}: degree = {degree}")

print(f"\nDegree Statistics:")
print(f"  Min degree: {min(degrees)}")
print(f"  Max degree: {max(degrees)}")
print(f"  Average degree: {sum(degrees)/len(degrees):.2f}")

## Degree Distribution Visualization

In [ ]:
# Create a bar chart of degree distribution
from collections import Counter

degree_counts = Counter(degrees)
degree_values = sorted(degree_counts.keys())
counts = [degree_counts[d] for d in degree_values]

fig = go.Figure(data=[
    go.Bar(
        x=[f"Degree {d}" for d in degree_values],
        y=counts,
        marker_color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'][:len(degree_values)],
        text=counts,
        textposition='auto'
    )
])

fig.update_layout(
    title='Degree Distribution of CellComplex Dual Graph',
    xaxis_title='Vertex Degree',
    yaxis_title='Number of Vertices',
    width=600,
    height=400
)

fig.show()

## Color Nodes by Degree

Let's create a visualization where node colors represent their connectivity (degree).

In [ ]:
def visualize_graph_by_degree(graph):
    """
    Visualize graph with nodes colored by their degree.
    """
    vertices = graph.Vertices()
    edges = graph.Edges()
    
    coords = [v.Coordinates() for v in vertices]
    degrees = [graph.VertexDegree(v) for v in vertices]
    
    fig = go.Figure()
    
    # Draw edges
    for edge in edges:
        edge_verts = edge.Vertices()
        if len(edge_verts) == 2:
            p1 = edge_verts[0].Coordinates()
            p2 = edge_verts[1].Coordinates()
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='rgba(100,100,100,0.5)', width=3),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # Draw vertices colored by degree
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    z = [c[2] for c in coords]
    
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(
            size=[8 + d * 2 for d in degrees],
            color=degrees,
            colorscale='Viridis',
            colorbar=dict(title='Degree'),
            line=dict(color='white', width=1)
        ),
        hovertext=[f"Node {i+1}<br>Degree: {d}" for i, d in enumerate(degrees)],
        hoverinfo='text',
        name='Graph Vertices'
    ))
    
    fig.update_layout(
        title='Graph Colored by Vertex Degree',
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=800,
        height=600
    )
    
    return fig

fig_degree = visualize_graph_by_degree(graph)
fig_degree.show()

## Adjacency Matrix Heatmap

Visualize the graph structure as an adjacency matrix.

In [ ]:
def create_adjacency_matrix(graph, labels):
    """
    Create and visualize the adjacency matrix of a graph.
    """
    vertices = graph.Vertices()
    n = len(vertices)
    
    # Create adjacency matrix
    adj_matrix = [[0] * n for _ in range(n)]
    
    for i, v in enumerate(vertices):
        adjacent = graph.AdjacentVertices(v)
        for adj_v in adjacent:
            # Find index of adjacent vertex
            adj_coords = adj_v.Coordinates()
            for j, v2 in enumerate(vertices):
                v2_coords = v2.Coordinates()
                if (abs(adj_coords[0] - v2_coords[0]) < 0.01 and
                    abs(adj_coords[1] - v2_coords[1]) < 0.01 and
                    abs(adj_coords[2] - v2_coords[2]) < 0.01):
                    adj_matrix[i][j] = 1
                    break
    
    # Create short labels
    short_labels = [l.replace('Cell ', 'C') for l in labels]
    
    fig = go.Figure(data=go.Heatmap(
        z=adj_matrix,
        x=short_labels,
        y=short_labels,
        colorscale=[[0, 'white'], [1, '#4ECDC4']],
        showscale=False
    ))
    
    fig.update_layout(
        title='Adjacency Matrix',
        xaxis=dict(title='', tickangle=45),
        yaxis=dict(title='', autorange='reversed'),
        width=700,
        height=600
    )
    
    return fig, adj_matrix

fig_adj, adj_matrix = create_adjacency_matrix(graph, cell_names)
fig_adj.show()

## Note on GraphViz Export

The original topologicpy notebook demonstrates exporting to GraphViz format (`.gv` files) for visualization with the GraphViz library. This feature is not currently available in topologic_fast.

**Features not available in topologic_fast:**
- `Graph.GraphVizGraph()` - Convert to GraphViz graph object
- `Graph.ExportToGraphVizGraph()` - Export to .gv file
- `Graph.EigenVectorCentrality()` - Calculate eigenvector centrality
- `Graph.PageRank()` - Calculate PageRank centrality
- `Graph.Compare()` - Compare two graphs
- Dictionary-based vertex/edge styling

However, you can achieve similar visualization results using Plotly as demonstrated above, or export the graph data to NetworkX for more advanced analysis.

In [ ]:
# Example: Export graph structure to NetworkX (if available)
try:
    import networkx as nx
    
    # Create NetworkX graph from topologic_fast graph
    G = nx.Graph()
    
    vertices = graph.Vertices()
    for i, v in enumerate(vertices):
        coords = v.Coordinates()
        G.add_node(i, pos=(coords[0], coords[1]), label=cell_names[i])
    
    # Add edges
    for i, v in enumerate(vertices):
        adjacent = graph.AdjacentVertices(v)
        for adj_v in adjacent:
            adj_coords = adj_v.Coordinates()
            for j, v2 in enumerate(vertices):
                v2_coords = v2.Coordinates()
                if (abs(adj_coords[0] - v2_coords[0]) < 0.01 and
                    abs(adj_coords[1] - v2_coords[1]) < 0.01 and
                    abs(adj_coords[2] - v2_coords[2]) < 0.01):
                    if i < j:  # Avoid duplicate edges
                        G.add_edge(i, j)
                    break
    
    # Calculate centrality measures
    eigenvector_cent = nx.eigenvector_centrality(G)
    pagerank = nx.pagerank(G)
    betweenness = nx.betweenness_centrality(G)
    
    print("Centrality Measures (via NetworkX):")
    print("=" * 60)
    print(f"{'Node':<10} {'Eigenvector':>12} {'PageRank':>12} {'Betweenness':>12}")
    print("-" * 60)
    for i in range(len(vertices)):
        print(f"{cell_names[i]:<10} {eigenvector_cent[i]:>12.4f} {pagerank[i]:>12.4f} {betweenness[i]:>12.4f}")
        
except ImportError:
    print("NetworkX not available. Install with: pip install networkx")
    print("NetworkX provides advanced graph analysis including centrality measures.")

## Summary

This notebook demonstrated:

1. **Creating a CellComplex** - A 3D grid of cells
2. **Deriving the Dual Graph** - Using `Graph.ByTopology()`
3. **2D and 3D Visualization** - Using Plotly
4. **Graph Analysis** - Degree distribution, adjacency matrix
5. **Integration with NetworkX** - For advanced centrality measures

### Key topologic_fast Graph Methods:
- `Graph.ByTopology(topology)` - Create graph from topology
- `graph.Order()` - Number of vertices
- `graph.Size()` - Number of edges
- `graph.Vertices()` - Get all vertices
- `graph.Edges()` - Get all edges
- `graph.VertexDegree(vertex)` - Get degree of a vertex
- `graph.AdjacentVertices(vertex)` - Get adjacent vertices
- `graph.Density()` - Graph density
- `graph.Diameter()` - Graph diameter
- `graph.IsConnected()` - Check connectivity